# SemDeDup — Semantic Redundancy Diagnostics

Analyse patch-level redundancy in the SSL training set using the SemDeDup
algorithm (Abbas et al., ICLR 2023). This notebook operates on the **same
embeddings** extracted by the Leiden clustering pipeline — no re-extraction
needed.

**Pipeline:**
1. Configuration
2. Load checkpoint & build encoder (or load cached embeddings)
3. Run SemDeDup redundancy profile
4. Summary table
5. Retention curve (SemDeDup Fig 3a)
6. Duplicate fraction curve (SemDeDup Fig 3b)
7. Within-cluster cosine histogram (SemDeDup Fig 3c)
8. Same-image vs cross-image cosine calibration
9. Per-cluster redundancy heatmap
10. Multi-k stability check
11. Per-image duplicate contribution

---
## 1 — Imports & setup

In [ ]:
%matplotlib inline

import os, sys, json, time, logging
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

In [ ]:
NB_DIR = Path.cwd().resolve()
ROOT   = NB_DIR
while ROOT.name != 'root' and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('ROOT:', ROOT)

In [ ]:
from training.config import ModelCfg
from training.seeding import seed_everything
from training.augment import ValSingleViewTransform
from training.data import TransformedSubset
from models import build_swin_encoder, count_params
from utils_data.patch_dataset import PatchDataset

from clustering.pipeline import extract_patch_embeddings, ClusterCfg
from clustering.semdedup import (
    SemDeDupCfg,
    redundancy_profile,
    print_summary,
    plot_retention_curve,
    plot_duplicate_fraction,
    plot_cosine_histogram,
    plot_same_vs_cross_image,
    plot_cluster_redundancy_heatmap,
    plot_stability,
)

print('All imports OK')

---
## 2 — Configuration

In [ ]:
from types import SimpleNamespace

cfg_nb = SimpleNamespace(
    # --- Paths ---
    encoder_ckpt      = '../outputs/REPLACE_ME/best_model.pt',
    data_root         = '../../data/patches_128_new2',
    output_dir        = '../outputs/semdedup_diagnostics',
    embedding_cache   = '../outputs/clustering_full/embeddings.npy',

    # --- Data ---
    exclude_patterns  = [],
    batch_size        = 64,
    num_workers       = 0,

    seed = 42,
)

In [ ]:
semdedup_cfg = SemDeDupCfg(
    seed=cfg_nb.seed,
    n_clusters=50,                       # main k for analysis
    stability_k_values=(25, 50, 100, 200),  # multi-k stability sweep
    epsilon_thresholds=(
        0.001, 0.005, 0.01, 0.02, 0.03, 0.05, 0.1, 0.2,
    ),
)
print(semdedup_cfg)

---
## 3 — Output directory, seed & device

In [ ]:
out_dir = Path(cfg_nb.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

seed_everything(cfg_nb.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Output:', out_dir)

---
## 4 — Load embeddings

Reuse the embedding cache from the Leiden clustering pipeline. If not
available, extract fresh embeddings from the encoder checkpoint.

In [ ]:
cache_path = Path(cfg_nb.embedding_cache)

if cache_path.exists():
    print(f'Loading cached embeddings from {cache_path}')
    d = np.load(cache_path, allow_pickle=True).item()
    Z             = d['Z'].astype(np.float32)
    filenames     = np.asarray(d['filenames'])
    source_images = np.asarray(d['source_images'])
    image_indices = np.asarray(d['image_indices'], dtype=np.int64)
    print(f'  Loaded {Z.shape[0]:,} patches, dim={Z.shape[1]}')
else:
    print('No embedding cache found — extracting from encoder checkpoint...')
    print(f'  Checkpoint: {cfg_nb.encoder_ckpt}')
    
    ds = PatchDataset(
        cfg_nb.data_root,
        channels=None,
        exclude_patterns=cfg_nb.exclude_patterns,
    )
    val_tf = ValSingleViewTransform(img_size=128)
    ds_tf  = TransformedSubset(ds, transform=val_tf)
    
    model_cfg = ModelCfg()
    encoder   = build_swin_encoder(model_cfg).to(device)
    ckpt      = torch.load(cfg_nb.encoder_ckpt, map_location=device, weights_only=False)
    encoder.load_state_dict(ckpt['encoder'], strict=False)
    print(f'  Encoder params: {count_params(encoder):,}')
    
    cluster_cfg = ClusterCfg(embedding_cache=str(cache_path))
    Z, filenames, source_images, image_indices = extract_patch_embeddings(
        encoder, ds_tf, device=device,
        batch_size=cfg_nb.batch_size,
        num_workers=cfg_nb.num_workers,
        cache_path=cache_path,
    )
    print(f'  Extracted & cached {Z.shape[0]:,} patches, dim={Z.shape[1]}')

print(f'\nUnique source images: {len(set(source_images))}')

---
## 5 — Run SemDeDup redundancy profile

This is the main analysis step. It:
- L2-normalises the raw encoder embeddings
- Runs k-means to bucket patches (SemDeDup §3)
- Computes within-cluster pairwise cosine similarities
- Sweeps ε thresholds and builds duplicate graphs
- Computes same-image vs cross-image cosine distributions
- Runs multi-k stability check

In [ ]:
t0 = time.time()
report = redundancy_profile(Z, source_images, semdedup_cfg)
elapsed = time.time() - t0
print(f'SemDeDup analysis completed in {elapsed:.1f}s')

---
## 6 — Summary table

In [ ]:
print_summary(report)

---
## 7 — Retention curve (SemDeDup Fig 3a)

Shows what fraction of the dataset would remain if semantic duplicates
at each ε were collapsed. Flat = few duplicates; steep = heavy redundancy.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_retention_curve(report, ax=ax)
fig.tight_layout()
fig.savefig(out_dir / 'retention_curve.png', dpi=150)
plt.show()

---
## 8 — Duplicate fraction curve (SemDeDup Fig 3b)

What percentage of patches have at least one semantic duplicate at each ε.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_duplicate_fraction(report, ax=ax)
fig.tight_layout()
fig.savefig(out_dir / 'duplicate_fraction.png', dpi=150)
plt.show()

---
## 9 — Within-cluster cosine histogram (SemDeDup Fig 3c)

Distribution of pairwise cosine similarities within k-means clusters.
A spike near 1.0 indicates heavy redundancy. Compare with SemDeDup
Figure 3c which shows LAION-440M has a large spike at cosine=1.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_cosine_histogram(report, ax=ax)
fig.tight_layout()
fig.savefig(out_dir / 'cosine_histogram.png', dpi=150)
plt.show()

---
## 10 — Same-image vs cross-image cosine calibration

SemDeDup's ε thresholds were calibrated on CLIP embeddings of natural
images. Our SSL encoder on fluorescence microscopy may have a very
different cosine distribution. This plot shows whether same-image
patch pairs (which share the same biological sample) are systematically
more similar than cross-image pairs — helping pick a meaningful ε.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_same_vs_cross_image(report, ax=ax)
fig.tight_layout()
fig.savefig(out_dir / 'same_vs_cross_image.png', dpi=150)
plt.show()

if len(report.same_image_cosines) > 0 and len(report.cross_image_cosines) > 0:
    print(f'Same-image  median cosine: {np.median(report.same_image_cosines):.4f}')
    print(f'Cross-image median cosine: {np.median(report.cross_image_cosines):.4f}')
    print(f'Same-image  95th pctl:     {np.percentile(report.same_image_cosines, 95):.4f}')
    print(f'Cross-image 95th pctl:     {np.percentile(report.cross_image_cosines, 95):.4f}')

---
## 11 — Per-cluster redundancy heatmap

Which k-means clusters are most internally similar? High mean cosine
= patches in that cluster are nearly identical (e.g., all-black
background patches clustered together).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
plot_cluster_redundancy_heatmap(report, ax=ax)
fig.tight_layout()
fig.savefig(out_dir / 'cluster_redundancy.png', dpi=150)
plt.show()

---
## 12 — Multi-k stability check

SemDeDup §6.1 shows that the choice of k is robust. We verify this
for our dataset by running k-means with different k values and checking
whether the estimated fraction remaining at the default ε is stable.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
plot_stability(report, ax=ax)
if ax is not None:
    fig.tight_layout()
    fig.savefig(out_dir / 'multi_k_stability.png', dpi=150)
plt.show()

---
## 13 — Per-image duplicate contribution

Which source images contribute the most redundant patches? Images
with many near-duplicate patches may be over-represented in training,
or may simply have large uniform regions (e.g., mostly background).

In [ ]:
# Sort images by duplicate fraction
img_stats = sorted(
    report.per_image_duplicate_frac.items(),
    key=lambda x: -x[1],
)

fig, ax = plt.subplots(figsize=(12, 5))
names = [s[0] for s in img_stats]
fracs = [s[1] * 100 for s in img_stats]
ax.barh(range(len(names)), fracs, color='C3', alpha=0.7)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=7)
ax.set_xlabel('% of patches that are duplicates')
ax.set_title(f'Per-image duplicate contribution (ε={report.default_epsilon})')
ax.invert_yaxis()
ax.grid(alpha=0.3, axis='x')
fig.tight_layout()
fig.savefig(out_dir / 'per_image_duplicates.png', dpi=150)
plt.show()

---
## 14 — Save report

Save the numeric results as a JSON for later reference.

In [ ]:
results = {
    'n_patches': report.n_patches,
    'n_clusters': report.n_clusters,
    'embedding_dim': report.embedding_dim,
    'default_epsilon': report.default_epsilon,
    'threshold_sweep': [
        {
            'epsilon': s.epsilon,
            'n_duplicate_pairs': s.n_duplicate_pairs,
            'n_patches_with_duplicate': s.n_patches_with_duplicate,
            'fraction_with_duplicate': round(s.fraction_with_duplicate, 4),
            'n_connected_components': s.n_connected_components,
            'n_unique_groups': s.n_unique_groups,
            'estimated_fraction_remaining': round(s.estimated_fraction_remaining, 4),
        }
        for s in report.threshold_stats
    ],
    'stability': report.stability_results,
    'per_image_duplicate_frac': {
        k: round(v, 4) for k, v in report.per_image_duplicate_frac.items()
    },
}

json_path = out_dir / 'semdedup_report.json'
with open(json_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Report saved to {json_path}')

---
## 15 — Interpretation guide

**How to read the results:**

| Plot | What to look for |
|------|------------------|
| Retention curve | Steep drop = heavy redundancy. Flat = unique patches |
| Duplicate fraction | >50% at ε=0.03 means extreme redundancy (like LAION) |
| Cosine histogram | Spike near 1.0 = many near-identical patches |
| Same vs cross-image | If same-image is much higher → biological sample bias |
| Cluster heatmap | High mean-cosine clusters = homogeneous groups (e.g. background) |
| Multi-k stability | Flat line = k choice doesn't matter (expected per SemDeDup §6.1) |
| Per-image duplicates | Images with >50% duplicates may need investigation |

**Choosing ε:**
- ε=0.001–0.005: only catches near-exact duplicates (perceptual)
- ε=0.01–0.03: semantic duplicates (same structure, different noise/crop)
- ε=0.05–0.1: semantically redundant (similar but visually distinct)
- ε>0.1: aggressive — may merge genuinely different morphologies

Use the same-vs-cross plot to calibrate: a good ε should sit at the
boundary where same-image cosine distribution starts to separate from
cross-image.